> # ⚠️ NOTEBOOK ARQUIVADO — NÃO EXECUTAR
>
> **Status:** obsoleto desde 03/jun/2026.
>
> Este notebook implementava o **piloto formal de decisão tipológica** —
> medir prevalência de misto e separabilidade de DN para decidir *entre*
> tipologia multi-rótulo e ternária mutuamente exclusiva. **Essa decisão já
> foi tomada:** a tipologia adotada é **ternária multi-rótulo**, com três
> flags positivos independentes (`epi_positivista`, `epi_interpretativa`,
> `epi_doutrinario_normativa`) e `postura_proeminente` derivada
> (`mixed` = positivista ∧ interpretativa, exclusivamente). Ver Addendum 1 ao
> pré-registro OSF e Manual v20 §0.1/§6.4.
>
> Toda a árvore de decisão abaixo (Blocos 4–6: separabilidade, silhouette,
> probe B1/B2, esquemas 3-A/3-B) está **morta** e usa o esquema antigo
> Chave A/B (`A_pos`, `A_int`, `B_label`, `B_forcing`) e o **Guia V2**, ambos
> descontinuados.
>
> **Substituído por:** `govai_calibracao_colab.ipynb` (fase de calibração
> inter-anotador, gate α_DN ≥ 0,40).
>
> Mantido apenas como registro histórico (replicabilidade / Open Science).

# 🔬 govAI — Piloto de tipologia epistemológica

**Objetivo:** medir (a) prevalência de misto e (c) separabilidade de DN para decidir
entre a tipologia multi-rótulo atual e a ternária mutuamente exclusiva (EE/IC/DN).

**Como usar este notebook:**
Clique no botão ▶ à esquerda de cada célula de código, na ordem de cima para baixo.
Espere cada célula terminar (o ▶ vira ✓) antes de passar para a próxima.

> ⚠️ **Antes de começar:** vá em `Runtime → Change runtime type → T4 GPU` e salve.
> Isso ativa a GPU gratuita, necessária para o passo de embeddings.


---
## BLOCO 0 — Configuração (rode uma vez por sessão)


In [ ]:
# 0-A: Montar o Google Drive (seus arquivos ficam salvos entre sessões)
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/govai_pilot'
os.makedirs(BASE, exist_ok=True)
print('Drive montado. Pasta do projeto:', BASE)


In [ ]:
# 0-B: Carregar o código do projeto a partir do zip
# Faça upload do arquivo govai_pilot_code.zip (baixado da sessão de trabalho)
from google.colab import files
import zipfile, shutil

print('Selecione govai_pilot_code.zip quando a caixa abrir abaixo:')
uploaded = files.upload()

zipfile.ZipFile('govai_pilot_code.zip').extractall('.')
print('Código extraído. Conteúdo da pasta pilot:',
      [f for f in os.listdir('pilot')])


In [ ]:
# 0-C: Instalar dependências (≈ 2-3 minutos na primeira vez)
!pip install pandas scikit-learn krippendorff requests transformers torch -q
print('Instalação concluída.')


---
## BLOCO 1 — Selecionar os artigos para o piloto

Você precisa de um arquivo **`corpus.csv`** com pelo menos as colunas:
- `doc_id` — identificador único de cada artigo
- colunas de estrato: ex. `cluster`, `area` (PA / IS), `region` (BR / intl)

Se ainda não tem o corpus em CSV, exporte de Notion / Zotero / Scopus com essas colunas.
O script vai sortear **150 artigos representativos** + **50 artigos de reforço de DN** (cluster Law).


In [ ]:
# 1-A: Fazer upload do corpus.csv
print('Selecione corpus.csv:')
uploaded = files.upload()
import shutil
shutil.copy('corpus.csv', f'{BASE}/corpus.csv')
print('corpus.csv salvo no Drive.')


In [ ]:
# 1-B: Rodar a seleção estratificada
!python pilot/sample_selection.py corpus.csv \
    --strata cluster area region \
    --n_prev 150 --n_boost 50 \
    --dn_filter 'cluster==Law' \
    --seed 42

import pandas as pd
sample = pd.read_csv('sample.csv')
print(sample.groupby('subsample').size())
shutil.copy('sample.csv', f'{BASE}/sample.csv')
print('sample.csv salvo no Drive.')


---
## BLOCO 2 — Baixar os abstracts

**Modo A (recomendado se você tem os DOIs):** prepare um `dois.csv` com coluna `doi`
e opcionalmente `doc_id`. Execute a célula 2-A.

**Modo B (filtro por periódico no OpenAlex):** edite o filtro na célula 2-B.

O resultado é `abstracts.csv` (doc_id, abstract).


In [ ]:
# 2-A: Por lista de DOIs  ← execute esta OU a 2-B, não as duas
print('Selecione dois.csv (colunas: doi e, opcionalmente, doc_id):')
uploaded = files.upload()
!python pilot/download_abstracts.py --dois dois.csv --mailto voce@fgv.br
shutil.copy('abstracts.csv', f'{BASE}/abstracts.csv')
import pandas as pd; df=pd.read_csv('abstracts.csv')
print(f'{len(df)} abstracts baixados. Primeiras linhas:'); print(df.head(2))


In [ ]:
# 2-B: Por filtro OpenAlex  ← execute esta OU a 2-A, não as duas
# Troque o filtro pelo periódico e período do seu corpus.
# Para achar o source.id: acesse https://api.openalex.org/sources?search=NOME_DO_PERIODICO
FILTRO = 'primary_location.source.id:S137773608,from_publication_date:2015-01-01'
!python pilot/download_abstracts.py --filter '{FILTRO}' --mailto voce@fgv.br
shutil.copy('abstracts.csv', f'{BASE}/abstracts.csv')
import pandas as pd; df=pd.read_csv('abstracts.csv')
print(f'{len(df)} abstracts baixados.'); print(df.head(2))


---
## BLOCO 3 — [OPCIONAL] Pré-classificação exploratória por LLM

Este bloco é uma **triagem provisória** — não substitui a anotação humana.
Serve para ter uma leitura rápida antes de anotar manualmente.

Para rodar, você precisa de uma **chave da API da Anthropic**.
Entre em https://console.anthropic.com → API Keys → copie a chave.


In [ ]:
# 3: Pré-classificação por LLM (triagem exploratória — NÃO é padrão-ouro)
from google.colab import userdata
import os

# Cole sua chave da API da Anthropic quando solicitado:
# Vá em Secrets (ícone 🔑 no painel esquerdo) → Add new secret → ANTHROPIC_API_KEY
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

!python pilot/llm_prepass.py abstracts.csv
shutil.copy('annotations_llm_prepass.csv', f'{BASE}/annotations_llm_prepass.csv')
print('Pronto. Resultado salvo no Drive (rotulado como exploratório).')


---
## ⏸️ PAUSA — Anotação humana

Agora você e o(a) segundo(a) anotador(a) anotam os artigos seguindo o
**Guia de Anotação V2 (guia_anotacao_posturas_v2.docx)**.

**Resultado esperado:** um arquivo `annotations.csv` com uma linha por
(doc_id, annotator) e as colunas:
```
doc_id, subsample, annotator, A_pos, A_int, B_label, B_forcing
```
Onde `annotator` é `ann1`, `ann2` ou `gold` (adjudicado).

Quando as anotações estiverem prontas, continue no Bloco 4.


---
## BLOCO 4 — Análise dos rótulos e concordância inter-anotador


In [ ]:
# 4: Carregar annotations.csv e rodar a análise de rótulos
print('Selecione annotations.csv:')
uploaded = files.upload()
shutil.copy('annotations.csv', f'{BASE}/annotations.csv')

!python pilot/run_labels.py annotations.csv

print()
print('--- Interpretação ---')
print('alpha (Chave B, DN vs. resto) >= 0.50 -> DN é coeso o suficiente para ser categoria plena.')
print('misto < 10% -> a exclusividade mútua descartaria pouca informação.')


---
## BLOCO 5 — Embeddings BERTimbau

> ⚠️ **Antes de rodar este bloco:** confirme que a GPU está ativa.
> `Runtime → Change runtime type → T4 GPU`. Se não estiver, o download e o
> processamento vão ser muito mais lentos (40-60 min vs ~5-10 min).

O BERTimbau (~430 MB) é baixado automaticamente do Hugging Face na primeira vez.
Isso pode levar 2-3 minutos.


In [ ]:
# 5: Gerar embeddings BERTimbau para os abstracts
!python pilot/embeddings.py abstracts.csv

shutil.copy('embeddings.npy', f'{BASE}/embeddings.npy')
shutil.copy('doc_ids.csv',    f'{BASE}/doc_ids.csv')

import numpy as np
X = np.load('embeddings.npy')
print(f'Embeddings gerados: {X.shape} (artigos × dimensão)')
print('Salvos no Drive.')


---
## BLOCO 6 — Separabilidade e decisão


In [ ]:
# 6: Separabilidade (silhouette, KMeans) + probe (B1 softmax vs B2 derivação) + decisão
!python pilot/run_analysis.py annotations.csv

print()
print('=== GUIA DE LEITURA DA DECISÃO ===')
print('Consulte os limiares pré-registrados em pilot/config.py:')
print('  misto < 10% AND alpha_DN >= 0.50 AND F1_DN(B1) >= F1_DN(B2) → ternária pura')
print('  misto >= 10% AND alpha_DN >= 0.50                            → primário + secundário (3-B)')
print('  alpha_DN < 0.50 OR F1_DN(B1) < F1_DN(B2)                   → manter multi-rótulo + derivação (3-A)')


In [ ]:
# 6b [alternativa sem GPU / sem BERTimbau]: usa TF-IDF como proxy
# Útil para uma primeira leitura rápida antes de gerar os embeddings reais.
!python pilot/run_analysis.py annotations.csv --tfidf abstracts.csv
print('(resultado com TF-IDF — exploratório; repita com BERTimbau para a decisão final)')


---
## BLOCO 7 — Baixar os resultados para o seu Mac

Todos os arquivos já foram salvos no seu Google Drive (pasta `govai_pilot`).
Para baixar diretamente do Colab, use a célula abaixo.


In [ ]:
# 7: Baixar arquivos de resultado
from google.colab import files
for fname in ['sample.csv','abstracts.csv','annotations_llm_prepass.csv',
              'embeddings.npy','doc_ids.csv']:
    try: files.download(fname)
    except: pass
print('Download iniciado para os arquivos disponíveis.')


---
## ❓ Problemas comuns

| Sintoma | Solução |
|---|---|
| `ModuleNotFoundError` | Rode novamente a célula 0-C (pip install) |
| `FileNotFoundError: corpus.csv` | Refaça o upload na célula correspondente |
| Célula demora mais de 10 min sem GPU | Ative a GPU: Runtime → Change runtime type → T4 GPU |
| `RuntimeError: CUDA out of memory` | Runtime → Disconnect and delete runtime → reconecte |
| `ANTHROPIC_API_KEY not found` | Adicione a chave em Secrets (ícone 🔑 no painel esquerdo) |
| Sessão expirou e arquivos sumiram | Os arquivos estão no Google Drive; recarregue com a célula 0-A |

---
*govAI · FAPESP Pós-doc · FGV EAESP · Fernando Leite*
